In [3]:
import json
import re


def parse_float_list(text):
    return [float(x.strip()) for x in text.split(",")]


folder = "../results/upper/"
input_file = folder + "upper_mid.txt"
output_file = folder + "upper_mid.json"

with open(input_file, "r", encoding="utf-8") as f:
    text = f.read()

# KL targets appear between epoch blocks
kl_targets = {
    int(epoch) + 1: float(target)
    for epoch, target in re.findall(
        r"Epoch\s+(\d+):\s+KL target set to\s+([-\deE+.]+)",
        text,
        flags=re.IGNORECASE,
    )
}

blocks = re.split(
    r"=+\s*Epoch\s+(\d+)\s*=+",
    text,
    flags=re.IGNORECASE,
)

epochs = []

for i in range(1, len(blocks), 2):
    epoch = int(blocks[i])
    block = blocks[i + 1]

    data = {"epoch": epoch}

    # --------------------------------------------------
    # Main metrics
    # --------------------------------------------------

    metrics = {
        "train_loss": r"Train loss:\s*([-\deE+.]+)",
        "val_nll": r"NLL on val set:\s*([-\deE+.]+)",
        "test_nll": r"NLL on test set:\s*([-\deE+.]+)",
        "posterior_val_nll": r"Posterior NLL on val:\s*([-\deE+.]+)",
        "posterior_train_nll": r"Posterior NLL on train:\s*([-\deE+.]+)",
        "prior_train_nll": r"Prior NLL on train:\s*([-\deE+.]+)",
    }

    for key, pattern in metrics.items():
        match = re.search(pattern, block, re.IGNORECASE)
        if match:
            data[key] = float(match.group(1))

    # --------------------------------------------------
    # Layer statistics
    # --------------------------------------------------

    array_patterns = {
        "layered_kl":
            r"Layered KL per layer:\s*\[([^\]]+)\]",
        "layer_budget":
            r"Layer budget per layer:\s*\[([^\]]+)\]",
        "layer_residual":
            r"Layer residual per layer:\s*\[([^\]]+)\]",
        "lambda":
            r"lambda_\s*per layer:\s*\[([^\]]+)\]",
    }

    for key, pattern in array_patterns.items():
        match = re.search(pattern, block, re.IGNORECASE)
        if match:
            data[key] = parse_float_list(match.group(1))

    # Handle scalar lambda values too
    if "lambda" not in data:
        match = re.search(
            r"lambda_\s*per layer:\s*([-\deE+.]+)",
            block,
            re.IGNORECASE,
        )
        if match:
            data["lambda"] = float(match.group(1))

    # --------------------------------------------------
    # Reconstruction / KL constraint
    # --------------------------------------------------

    match = re.search(
        r"reconstruction=([-\deE+.]+)\s+kl_constraint=([-\deE+.]+)",
        block,
        re.IGNORECASE,
    )

    if match:
        data["reconstruction"] = float(match.group(1))
        data["kl_constraint"] = float(match.group(2))

    # --------------------------------------------------
    # Reconstruction / KL loss (for warmup runs)
    # --------------------------------------------------

    match = re.search(
        r"reconstruction=([-\deE+.]+)\s+kl_loss=([-\deE+.]+)",
        block,
        re.IGNORECASE,
    )

    if match:
        data["reconstruction"] = float(match.group(1))
        data["kl_loss"] = float(match.group(2))

    # --------------------------------------------------
    # Posterior / Prior latent statistics
    # --------------------------------------------------

    match = re.search(
        r"posterior mean=([-\deE+.]+)\s+"
        r"logvar=([-\deE+.]+)\s+"
        r"prior mean=([-\deE+.]+)\s+"
        r"prior logvar=([-\deE+.]+)",
        block,
        re.IGNORECASE,
    )

    if match:
        data["posterior_mean"] = float(match.group(1))
        data["posterior_logvar"] = float(match.group(2))
        data["prior_mean"] = float(match.group(3))
        data["prior_logvar"] = float(match.group(4))

    # --------------------------------------------------
    # Scrabbled NLL
    # --------------------------------------------------

    match = re.search(
        r"Scrabbled NLL on train:\s*([-\deE+.]+),\s*"
        r"Scrabbled NLL on val:\s*([-\deE+.]+)",
        block,
        re.IGNORECASE,
    )

    if match:
        data["scrabbled_train_nll"] = float(match.group(1))
        data["scrabbled_val_nll"] = float(match.group(2))

    # --------------------------------------------------
    # Trial validation loss
    # --------------------------------------------------

    match = re.search(
        r"Trial\s+(\d+):\s+Epoch\s+\d+:\s+Validation loss\s+([-\deE+.]+)",
        block,
        re.IGNORECASE,
    )

    if match:
        data["trial"] = int(match.group(1))
        data["trial_validation_loss"] = float(match.group(2))

    # --------------------------------------------------
    # KL target
    # --------------------------------------------------

    if epoch in kl_targets:
        data["kl_target"] = kl_targets[epoch]

    epochs.append(data)

with open(output_file, "w", encoding="utf-8") as f:
    json.dump(epochs, f, indent=4)

print(f"Parsed {len(epochs)} epochs")
print(f"Saved to {output_file}")

Parsed 386 epochs
Saved to ../results/upper/upper_mid.json
